In [1]:
import os
import sys
import json
from pathlib import Path
from dotenv import load_dotenv

# --- SET YOUR PROJECT CONFIG HERE (Overrides .env) ---
os.environ["VERTEXAI_PROJECT"] = ""
os.environ["VERTEXAI_LOCATION"] = "us-central1"
os.environ["LLM_BACKEND"] = "vertex_ai"
os.environ["MODEL_NAME"] = "gemini-2.5-flash"
os.environ["CREWAI_TRACING_ENABLED"] = "false"

from crewai import Agent, Task, Crew, Process

# Import local configurations and tools
from config.llm_config import get_llm
from tools.crewai_tools import (
    ParseDocxTool,
    ExtractProtocolTablesTool,
    DetectTemplateSignalsTool,
)

# --- 1. SET PROJECT CONFIG ---
os.environ["VERTEXAI_PROJECT"] = ""
os.environ["VERTEXAI_LOCATION"] = "us-central1"
os.environ["LLM_BACKEND"] = "vertex_ai"
os.environ["MODEL_NAME"] = "gemini-2.5-flash-lite"
os.environ["CREWAI_TRACING_ENABLED"] = "false"

os.environ["LITELLM_NUM_RETRIES"] = "5"            # If it gets a 429, try 5 more times
os.environ["LITELLM_INITIAL_BACKOFF"] = "10.0"     # Wait 10 seconds before the first retry
os.environ["LITELLM_MAX_BACKOFF"] = "60.0"         # Don't wait longer than 60 seconds

# Save the original Jupyter stream before CrewAI imports anything
original_jupyter_stdout = sys.stdout

base_dir = Path(os.getcwd())
load_dotenv(base_dir / ".env")

print("--- Environment Variables ---")
print(f"MODEL_NAME={os.getenv('MODEL_NAME')} on {os.getenv('VERTEXAI_PROJECT')}")

In [2]:
def build_first_demo_crew() -> Crew:
    llm = get_llm()

    parse_docx_tool = ParseDocxTool()
    extract_tables_tool = ExtractProtocolTablesTool()
    detect_signals_tool = DetectTemplateSignalsTool()

    document_parser_agent = Agent(
        role="Clinical Document Parser",
        goal="Parse protocol and SAP template documents into structured JSON artifacts.",
        backstory=(
            "You are meticulous about document structure, formatting, and reproducible parsing. "
            "You rely on your tools to process the files accurately."
        ),
        llm=llm,
        tools=[parse_docx_tool],
        verbose=True,
        allow_delegation=False,
        max_rpm=2,
    )

    signal_detection_agent = Agent(
        role="Template Signal Analyst",
        goal="Extract protocol tables and detect SAP template update/remove signals.",
        backstory=(
            "You specialize in converting parsed clinical documents into structured workflow artifacts, "
            "including table extractions and update-unit lists."
        ),
        llm=llm,
        tools=[extract_tables_tool, detect_signals_tool],
        verbose=True,
        allow_delegation=False,
        max_rpm=2,
    )

    # ==========================================
    # TODO 1: Create the QA Reviewer Agent
    # ==========================================
    # Instructions: Define a new Agent named `qa_reviewer_agent`.
    # - role: "Clinical QA Reviewer"
    # - goal: "Review the detected update units for clarity and clinical formatting."
    # - backstory: "You are a senior clinical data manager with a sharp eye for errors."
    # - llm: llm
    # - verbose: True
    # - allow_delegation: False
    
    qa_reviewer_agent = Agent(
        # FILL IN YOUR CODE HERE
    )

    workflow_reporter_agent = Agent(
        role="Workflow Reporter",
        goal="Summarize the generated artifacts and explain what the demo accomplished.",
        backstory=(
            "You create concise, accurate summaries of automation outputs for training and demonstration."
        ),
        llm=llm,
        verbose=True,
        allow_delegation=False,
        max_rpm=2,
    )

    parse_protocol_task = Task(
        description=(
            "Use the parse_docx_tool to parse the protocol DOCX file.\n"
            "- file_path: {protocol_path}\n"
            "- output_json_path: {protocol_json_path}\n"
            "Return a short confirmation that includes paragraph count and table count."
        ),
        expected_output="A short confirmation JSON-like summary for the parsed protocol file.",
        agent=document_parser_agent,
    )

    parse_template_task = Task(
        description=(
            "Use the parse_docx_tool to parse the SAP template DOCX file.\n"
            "- file_path: {template_path}\n"
            "- output_json_path: {template_json_path}\n"
            "Return a short confirmation that includes paragraph count and table count."
        ),
        expected_output="A short confirmation JSON-like summary for the parsed SAP template file.",
        agent=document_parser_agent,
    )

    extract_tables_task = Task(
        description=(
            "Use the extract_protocol_tables_tool on the parsed protocol JSON.\n"
            "- parsed_protocol_json_path: {protocol_json_path}\n"
            "- output_json_path: {protocol_tables_json_path}\n"
            "Return a short confirmation with the number of extracted tables."
        ),
        expected_output="A short confirmation JSON-like summary for extracted protocol tables.",
        agent=signal_detection_agent,
        context=[parse_protocol_task],
    )

    detect_signals_task = Task(
        description=(
            "Use the detect_template_signals_tool on the parsed template JSON.\n"
            "- parsed_template_json_path: {template_json_path}\n"
            "- output_json_path: {update_units_json_path}\n"
            "Current signal rules: blue text = update placeholder, green text = guidance/remove.\n"
            "Return a short confirmation with the number of detected update units."
        ),
        expected_output="A short confirmation JSON-like summary for detected template update units.",
        agent=signal_detection_agent,
        context=[parse_template_task],
    )

    # ==========================================
    # TODO 2: Create the QA Task
    # ==========================================
    # Instructions: Define a new Task named `qa_review_task`.
    # - description: "Review the detected update units from the previous task. Add a short QA approval note."
    # - expected_output: "A short QA review report stating if the extracted units look reasonable."
    # - agent: qa_reviewer_agent
    # - context: [detect_signals_task]  <-- This ensures it waits for the signals to be detected!
    
    qa_review_task = Task(
        # FILL IN YOUR CODE HERE
    )

    summarize_task = Task(
        description=(
            "Summarize what artifacts were created in this first demo. "
            "Use the prior task results (including the QA review) and produce a concise operational summary "
            "that explains what the CrewAI version accomplished."
        ),
        expected_output=(
            "A concise summary of the generated outputs, counts, and the purpose of this first demo."
        ),
        agent=workflow_reporter_agent,
        context=[
            parse_protocol_task,
            parse_template_task,
            extract_tables_task,
            detect_signals_task,
            qa_review_task, 
        ],
    )

    return Crew(
        agents=[
            document_parser_agent,
            signal_detection_agent,
            qa_reviewer_agent,
            workflow_reporter_agent,
        ],
        tasks=[
            parse_protocol_task,
            parse_template_task,
            extract_tables_task,
            detect_signals_task,
            qa_review_task,
            summarize_task,
        ],
        process=Process.sequential,
        verbose=True,
        max_rpm=2,
    )

In [3]:
import os
import json
from pathlib import Path

from dotenv import load_dotenv

In [4]:
def main() -> None:
    """Run the first CrewAI demo."""
    # Load .env from the current project directory
    base = Path(os.getcwd())
    load_dotenv(base / ".env")

    # Optional debug prints
    print(f"LLM_BACKEND={os.getenv('LLM_BACKEND')}")
    print(f"MODEL_NAME={os.getenv('MODEL_NAME')}")

    data_dir = base / "data"
    out_dir = base / "outputs"
    out_dir.mkdir(parents=True, exist_ok=True)

    inputs = {
        "protocol_path": str(data_dir / "protocol.docx"),
        "template_path": str(data_dir / "sap_template.docx"),
        "protocol_json_path": str(out_dir / "protocol_parsed.json"),
        "template_json_path": str(out_dir / "sap_template_parsed.json"),
        "protocol_tables_json_path": str(out_dir / "protocol_tables.json"),
        "update_units_json_path": str(out_dir / "update_units.json"),
    }

    crew = build_first_demo_crew()
    try:
        result = crew.kickoff(inputs=inputs)
    finally:
        # --- 2. THE HARD RESET ---
        # Force the stream back to Jupyter's original stream
        sys.stdout = original_jupyter_stdout
        
    summary_path = out_dir / "crewai_demo_result.txt"
    summary_path.write_text(str(result), encoding="utf-8")

    deterministic_summary = {
        "protocol_json_path": inputs["protocol_json_path"],
        "template_json_path": inputs["template_json_path"],
        "protocol_tables_json_path": inputs["protocol_tables_json_path"],
        "update_units_json_path": inputs["update_units_json_path"],
        "crewai_result_path": str(summary_path),
    }

    (out_dir / "crewai_demo_summary.json").write_text(
        json.dumps(deterministic_summary, indent=2),
        encoding="utf-8",
    )

    print("CrewAI first demo completed.")
    print(f"Result summary saved to: {summary_path}")

In [ ]:
if __name__ == "__main__":
    main()